In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [2]:
from google.cloud import bigquery
import pandas as pd

# Inicializa el cliente de BigQuery
client = bigquery.Client(project='dataton-2024-team-01-cofares')

# Ejecuta la consulta y convierte los datos en un DataFrame de Pandas desde BigQuery datos_no_descriptions_eans
query = "SELECT * FROM `dataton-2024-team-01-cofares.datos_cofares.muestra_temporal`"
df = client.query(query).to_dataframe()
print(df.head())

# Leer el archivo df_img_description.parquet para utilizarlo como df
#df = pd.read_parquet("df_img_description.parquet")
#print(df.head())

                                                 uri  \
0  gs://dataton-2024-team-01-cofares-datastore/im...   
1  gs://dataton-2024-team-01-cofares-datastore/im...   
2  gs://dataton-2024-team-01-cofares-datastore/im...   
3  gs://dataton-2024-team-01-cofares-datastore/im...   
4  gs://dataton-2024-team-01-cofares-datastore/im...   

                             ml_generate_text_result  \
0  {'candidates': [{'avg_logprobs': -0.2053625128...   
1  {'candidates': [{'avg_logprobs': -0.3342012763...   
2  {'candidates': [{'avg_logprobs': -0.2806888818...   
3  {'candidates': [{'avg_logprobs': -0.2440256327...   
4  {'candidates': [{'avg_logprobs': -0.2973315341...   

                                         descripcion forma color  \
0  ```json\n{"forma": "rectangular", "color": ["b...  None  None   
1  ```json\n{"forma": "rectangular", "color": ["b...  None  None   
2  ```json\n{"forma": "rectangular", "color": ["b...  None  None   
3  ```json\n{"forma": "rectangular", "color": "ro...  

/Users/gabrielnoguera/Documents/DataHub/app-flask/cofaresapp/.venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/Users/gabrielnoguera/Documents/DataHub/app-flask/cofaresapp/.venv/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:207: UserWarning: Unable to determine Arrow type for field 'ml_generate_text_result'.
  warnings.warn(


In [5]:
import json

# Verificar los datos antes de procesarlos
print("Ejemplo de dato crudo:")
print(df['descripcion'].iloc[0])
print("\nTipo de dato:", type(df['descripcion'].iloc[0]))

# Intenta limpiar y procesar una sola fila primero
muestra = df['descripcion'].iloc[0]
muestra_limpia = muestra.replace('```json\n', '').replace('\n```', '')
print("\nDespués de limpieza:")
print(muestra_limpia)

try:
    # Intenta parsear la muestra
    dict_muestra = json.loads(muestra_limpia)
    print("\nParseo exitoso:")
    print(dict_muestra)
except json.JSONDecodeError as e:
    print("\nError al parsear JSON:")
    print(e)

Ejemplo de dato crudo:
{"forma": "rectangular", "color": ["blanco", "azul", "dorado", "rosa"], "descripcion_visual": "Una caja de cartón con una imagen de un ojo y una gota de líquido azul. La caja tiene el nombre de la marca 'abéñula' y la palabra 'azul'.", "empaque": "caja", "zona_de_aplicacion": "ojos"}

Tipo de dato: <class 'str'>

Después de limpieza:
{"forma": "rectangular", "color": ["blanco", "azul", "dorado", "rosa"], "descripcion_visual": "Una caja de cartón con una imagen de un ojo y una gota de líquido azul. La caja tiene el nombre de la marca 'abéñula' y la palabra 'azul'.", "empaque": "caja", "zona_de_aplicacion": "ojos"}

Parseo exitoso:
{'forma': 'rectangular', 'color': ['blanco', 'azul', 'dorado', 'rosa'], 'descripcion_visual': "Una caja de cartón con una imagen de un ojo y una gota de líquido azul. La caja tiene el nombre de la marca 'abéñula' y la palabra 'azul'.", 'empaque': 'caja', 'zona_de_aplicacion': 'ojos'}


In [7]:
import json

def parse_json_safely(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"Error en el texto: {text[:150]}...")  # Muestra los primeros 150 caracteres
        print(f"Error específico: {str(e)}")
        return {}

# Intentemos procesar fila por fila para ver dónde está el error
for i in range(5):  # Revisamos las primeras 5 filas
    print(f"\nFila {i}:")
    print(df['descripcion'].iloc[i])
    result = parse_json_safely(df['descripcion'].iloc[i])
    print("Resultado:", result)



Fila 0:
{"forma": "rectangular", "color": ["blanco", "azul", "dorado", "rosa"], "descripcion_visual": "Una caja de cartón con una imagen de un ojo y una gota de líquido azul. La caja tiene el nombre de la marca 'abéñula' y la palabra 'azul'.", "empaque": "caja", "zona_de_aplicacion": "ojos"}
Resultado: {'forma': 'rectangular', 'color': ['blanco', 'azul', 'dorado', 'rosa'], 'descripcion_visual': "Una caja de cartón con una imagen de un ojo y una gota de líquido azul. La caja tiene el nombre de la marca 'abéñula' y la palabra 'azul'.", 'empaque': 'caja', 'zona_de_aplicacion': 'ojos'}

Fila 1:
{"forma": "rectangular", "color": ["blanco", "azul", "negro", "dorado", "rojo", "rosa"], "descripcion_visual": "Una caja de cartón con una imagen de un ojo y una pequeña caja de color azul con la marca abéñula. La caja tiene un fondo rosa y la parte superior es de color dorado con la marca abéñula en letras rojas. La caja tiene un texto en español que dice \"Vida en tus ojos\" y \"Tamaño pequeño\".

In [8]:
import json

def clean_and_parse_json(text):
    try:
        # Si el texto parece estar truncado, intentamos repararlo
        if text.count('{') > text.count('}'):
            text = text + '"}'  # Cerramos el string y el objeto JSON
        
        # Intentamos parsear
        return json.loads(text)
    except json.JSONDecodeError:
        # Si falla, retornamos un diccionario con valores None
        return {
            'forma': None,
            'color': None,
            'descripcion_visual': None,
            'empaque': None,
            'zona_de_aplicacion': None
        }

# Aplicar la función a cada fila
df['descripcion'] = df['descripcion'].apply(clean_and_parse_json)

# Extraer cada campo del diccionario a su respectiva columna
df['forma'] = df['descripcion'].apply(lambda x: x.get('forma'))
df['color'] = df['descripcion'].apply(lambda x: x.get('color'))
df['descripcion_visual'] = df['descripcion'].apply(lambda x: x.get('descripcion_visual'))
df['empaque'] = df['descripcion'].apply(lambda x: x.get('empaque'))
df['zona_de_aplicacion'] = df['descripcion'].apply(lambda x: x.get('zona_de_aplicacion'))

# Opcional: eliminar la columna original 'descripcion'
# df.drop('descripcion', axis=1, inplace=True)

print(df.head())

                                                 uri  \
0  gs://dataton-2024-team-01-cofares-datastore/im...   
1  gs://dataton-2024-team-01-cofares-datastore/im...   
2  gs://dataton-2024-team-01-cofares-datastore/im...   
3  gs://dataton-2024-team-01-cofares-datastore/im...   
4  gs://dataton-2024-team-01-cofares-datastore/im...   

                             ml_generate_text_result  \
0  {'candidates': [{'avg_logprobs': -0.2053625128...   
1  {'candidates': [{'avg_logprobs': -0.3342012763...   
2  {'candidates': [{'avg_logprobs': -0.2806888818...   
3  {'candidates': [{'avg_logprobs': -0.2440256327...   
4  {'candidates': [{'avg_logprobs': -0.2973315341...   

                                         descripcion        forma  \
0  {'forma': 'rectangular', 'color': ['blanco', '...  rectangular   
1  {'forma': 'rectangular', 'color': ['blanco', '...  rectangular   
2  {'forma': 'rectangular', 'color': ['blanco', '...  rectangular   
3  {'forma': 'rectangular', 'color': 'rosa, azul,.